In [1]:
import numpy as np
import sounddevice as sd
from faster_whisper import WhisperModel
import collections
import time

# --- 설정 (필요에 따라 조정하세요) ---
SAMPLE_RATE = 16000         # 샘플 속도
CHUNK_DURATION = 1.0        # 청크 길이 (초)
VOLUME_THRESHOLD = 0.005      # 볼륨 임계값 (마이크 환경에 맞게 조정 필요!)
SILENCE_DURATION = 2.0      # 정적 시간 (초)

# --- 모델 로딩 ---
print("모델 로딩 중... (large-v3, cuda)")
try:
    asr_model = WhisperModel("large-v3", device="cuda", compute_type="float16")
    print("모델 로딩 완료.")
except Exception as e:
    print(f"CUDA 모델 로딩 실패: {e}")
    print("CPU로 모델을 로딩합니다.")
    asr_model = WhisperModel("large-v3", device="cpu", compute_type="float32")
    print("모델 로딩 완료.")


class RealTimeTranscriber:
    def __init__(self):
        self.audio_buffer = collections.deque()
        self.is_recording = False
        self.silent_chunks = 0
        self.max_silent_chunks = int(SILENCE_DURATION / CHUNK_DURATION)
        self.chunk_samples = int(SAMPLE_RATE * CHUNK_DURATION)

    def _process_audio_chunk(self, indata, frames, time, status):
        """sounddevice InputStream의 콜백 함수"""
        if status:
            print(status, flush=True)

        volume = np.sqrt(np.mean(indata**2))

        if self.is_recording:
            # 녹음 중일 때
            self.audio_buffer.append(indata.copy())
            if volume < VOLUME_THRESHOLD:
                self.silent_chunks += 1
                print(".", end="", flush=True)
                if self.silent_chunks >= self.max_silent_chunks:
                    self.is_recording = False  # 녹음 종료 플래그 설정
                    print("\n녹음 종료")
            else:
                self.silent_chunks = 0
                print("🔊", end="", flush=True)
        else:
            # 녹음 대기 중일 때
            if volume > VOLUME_THRESHOLD:
                print("녹음 시작...")
                self.is_recording = True
                self.silent_chunks = 0
                self.audio_buffer.clear()
                self.audio_buffer.append(indata.copy()) # **수정된 부분: 첫 소리도 버퍼에 추가**

    def _transcribe(self, audio_data):
        """버퍼에 쌓인 오디오 데이터를 텍스트로 변환"""
        try:
            print("음성 인식 중...")
            segments, _ = asr_model.transcribe(audio_data, language="ko", vad_filter=True)
            
            texts = [seg.text.strip() for seg in segments]
            result = " ".join(texts).strip()

            if result:
                print(f"결과: {result}")
                return result
            else:
                print("인식된 텍스트 없음")
                return None
        except Exception as e:
            print(f"음성 인식 오류: {e}")
            import traceback
            traceback.print_exc()
            return None

    def start(self):
        """실시간 음성 인식 시작"""
        print("마이크 입력을 시작합니다. 말씀해주세요...")
        # InputStream을 사용하여 비동기적으로 오디오를 받습니다.
        with sd.InputStream(samplerate=SAMPLE_RATE,
                             channels=1,
                             dtype=np.float32,
                             blocksize=self.chunk_samples,
                             callback=self._process_audio_chunk):
            while True:
                if not self.is_recording and len(self.audio_buffer) > 0:
                    # 녹음이 종료되고 버퍼에 데이터가 있으면 처리 시작
                    full_audio = np.concatenate(list(self.audio_buffer), axis=0)
                    self.audio_buffer.clear()
                    
                    full_audio = np.squeeze(full_audio) # (N, 1) -> (N,) 형태로 변경
                    
                    result = self._transcribe(full_audio)

                    if result and ("종료" in result or "끝" in result):
                        print("종료 명령어를 감지하여 프로그램을 종료합니다.")
                        break
                
                time.sleep(0.1) # CPU 사용률을 낮추기 위해 잠시 대기


if __name__ == "__main__":
    transcriber = RealTimeTranscriber()
    transcriber.start()

모델 로딩 중... (large-v3, cuda)
모델 로딩 완료.
마이크 입력을 시작합니다. 말씀해주세요...
녹음 시작...
🔊..
녹음 종료
음성 인식 중...
결과: 한강책 있니
녹음 시작...
..
녹음 종료
음성 인식 중...
인식된 텍스트 없음


KeyboardInterrupt: 

In [ ]:
import torch
print(torch.version.cuda)  
print(torch.cuda.is_available())  

11.8
True
